In [ ]:
#pip install notebook requests beautifulsoup4 pandas lxml scrapy

##1- Fast dataset creation **(Requests + BeautifulSoup)** — inside Notebook

This is the fastest method for small/medium sites and perfect for learning.

Perform Web Scraping on the following website:
 https://books.toscrape.com/catalogue/page-1.html  

Cell 1 — **imports**

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [10]:
base_url = "https://books.toscrape.com/catalogue/page-{}.html"
books_data = []

Cell 3 — **scrape ALL pages automatically**

In [11]:
for page in range(1, 51):
    print(f"Scraping Page {page}...")
    
    url = base_url.format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    books = soup.find_all("article", class_="product_pod")
    
    for book in books:
        
        
        book_name = book.h3.a["title"]
        
        
        price = book.find("p", class_="price_color").text.strip()
        
        
        stock = book.find("p", class_="instock availability").text.strip()
        
        
        relative_link = book.h3.a["href"]
        book_link = "https://books.toscrape.com/catalogue/" + relative_link.replace("../", "")
        
        
        book_response = requests.get(book_link)
        book_soup = BeautifulSoup(book_response.text, "html.parser")
        
        
        description_tag = book_soup.find("meta", attrs={"name": "description"})
        
        if description_tag:
            description = description_tag["content"].strip()
        else:
            description = "No description"
        
        books_data.append({
            "book_name": book_name,
            "book_description": description,
            "book_price": price,
            "stock": stock,
            "page": page
        })

Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...
Scraping Page 8...
Scraping Page 9...
Scraping Page 10...
Scraping Page 11...
Scraping Page 12...
Scraping Page 13...
Scraping Page 14...
Scraping Page 15...
Scraping Page 16...
Scraping Page 17...
Scraping Page 18...
Scraping Page 19...
Scraping Page 20...
Scraping Page 21...
Scraping Page 22...
Scraping Page 23...
Scraping Page 24...
Scraping Page 25...
Scraping Page 26...
Scraping Page 27...
Scraping Page 28...
Scraping Page 29...
Scraping Page 30...
Scraping Page 31...
Scraping Page 32...
Scraping Page 33...
Scraping Page 34...
Scraping Page 35...
Scraping Page 36...
Scraping Page 37...
Scraping Page 38...
Scraping Page 39...
Scraping Page 40...
Scraping Page 41...
Scraping Page 42...
Scraping Page 43...
Scraping Page 44...
Scraping Page 45...
Scraping Page 46...
Scraping Page 47...
Scraping Page 48...
Scraping Page 49...
Scraping Page 50...


Cell 4 — **convert to dataset**

In [12]:
df = pd.DataFrame(books_data)
df.head()

,book_name,book_description,book_price,stock,page
0,A Light in the Attic,It's hard to imagine a world without A Light i...,Â£51.77,In stock,1
1,Tipping the Velvet,"""Erotic and absorbing...Written with starling ...",Â£53.74,In stock,1
2,Soumission,"Dans une France assez proche de la nÃ´tre, un ...",Â£50.10,In stock,1
3,Sharp Objects,"WICKED above her hipbone, GIRL across her hear...",Â£47.82,In stock,1
4,Sapiens: A Brief History of Humankind,From a renowned historian comes a groundbreaki...,Â£54.23,In stock,1


Cell 5 — **export CSV (FINAL DATASET)**

In [13]:
df.to_csv("books_data.csv", index=False)

In [14]:
df.shape

(1000, 5)

You now created a **real dataset from the web in under 3 minutes.**

1- **Create Scrapy Project**

In [ ]:
!scrapy startproject quotes_project

2- **Move Into Project Folder**

In [8]:
%cd quotes_project

d:\HNU\Sections\Third year Second Semester\Deep learning\Section 2\quotes_project


3- **Create Spider**

In [ ]:
!scrapy genspider books books.toscrape.com

Created spider 'quotes' using template 'basic' in module:
  quotes_project.spiders.quotes


4- **Replace Spider Code**

In [ ]:
import scrapy

class BooksSpider(scrapy.Spider):
    name = "books"
    allowed_domains = ["books.toscrape.com"]
    
    start_urls = [
        f"https://books.toscrape.com/catalogue/page-{i}.html"
        for i in range(1, 51)
    ]

    def parse(self, response):
        books = response.css("article.product_pod")
        
        for book in books:
            
            book_name = book.css("h3 a::attr(title)").get()
            price = book.css("p.price_color::text").get()
            stock = book.css("p.instock.availability::text").getall()
            stock = "".join(stock).strip()
            
            book_link = book.css("h3 a::attr(href)").get()
            book_link = response.urljoin(book_link)
            
            yield scrapy.Request(
                book_link,
                callback=self.parse_book,
                meta={
                    "book_name": book_name,
                    "price": price,
                    "stock": stock,
                    "page": response.url.split("-")[-1].split(".")[0]
                }
            )

    def parse_book(self, response):
        description = response.css("meta[name='description']::attr(content)").get()
        
        yield {
            "book_name": response.meta["book_name"],
            "book_description": description,
            "book_price": response.meta["price"],
            "stock": response.meta["stock"],
            "page": response.meta["page"]
        }

Spider updated successfully!


5- **Run Spider & Export CSV**

In [ ]:
!scrapy crawl books -o books_scrapy.csv